# Step 1 — Initial Data Audit

In [2]:
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

In [3]:
# Load your dataset
df = pd.read_csv("postings.csv")  
print("ROWS, COLUMNS:", df.shape)

ROWS, COLUMNS: (123849, 13)


In [4]:
print("\nCOLUMNS:\n", df.columns.tolist())


COLUMNS:
 ['title', 'description', 'skills_desc', 'formatted_work_type', 'max_salary', 'med_salary', 'min_salary', 'location', 'pay_period', 'formatted_experience_level', 'remote_allowed', 'work_type', 'normalized_salary']


In [5]:
print("\nSAMPLE ROWS:")
df.head()


SAMPLE ROWS:


,title,description,skills_desc,formatted_work_type,max_salary,med_salary,min_salary,location,pay_period,formatted_experience_level,remote_allowed,work_type,normalized_salary
0,Marketing Coordinator,Job descriptionA leading real estate firm in N...,Requirements: \n\nWe are seeking a College or ...,Full-time,20.0,NaN,17.0,"Princeton, NJ",HOURLY,NaN,NaN,FULL_TIME,38480.0
1,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",NaN,Full-time,50.0,NaN,30.0,"Fort Collins, CO",HOURLY,NaN,NaN,FULL_TIME,83200.0
2,Assitant Restaurant Manager,The National Exemplar is accepting application...,We are currently accepting resumes for FOH - A...,Full-time,65000.0,NaN,45000.0,"Cincinnati, OH",YEARLY,NaN,NaN,FULL_TIME,55000.0
3,Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,This position requires a baseline understandin...,Full-time,175000.0,NaN,140000.0,"New Hyde Park, NY",YEARLY,NaN,NaN,FULL_TIME,157500.0
4,Service Technician,Looking for HVAC service tech with experience ...,NaN,Full-time,80000.0,NaN,60000.0,"Burlington, IA",YEARLY,NaN,NaN,FULL_TIME,70000.0


In [6]:
print("\nDTYPES:")
print(df.dtypes)


DTYPES:
title                          object
description                    object
skills_desc                    object
formatted_work_type            object
max_salary                    float64
med_salary                    float64
min_salary                    float64
location                       object
pay_period                     object
formatted_experience_level     object
remote_allowed                float64
work_type                      object
normalized_salary             float64
dtype: object


In [7]:
print("\nMISSING VALUES (%):")
missing = df.isna().mean() * 100
missing[missing > 0].sort_values(ascending=False).round(2)


MISSING VALUES (%):


skills_desc                   98.03
med_salary                    94.93
remote_allowed                87.69
max_salary                    75.94
min_salary                    75.94
pay_period                    70.87
normalized_salary             70.87
formatted_experience_level    23.75
description                    0.01
dtype: float64

In [8]:
print("\nUNIQUE COUNTS (selected columns):")
for c in ['formatted_experience_level','work_type','remote_allowed','pay_period','location']:
    if c in df.columns:
        print(f"\n- {c}: {df[c].nunique()} unique values")
        print(df[c].value_counts(dropna=False).head(10))


UNIQUE COUNTS (selected columns):

- formatted_experience_level: 6 unique values
formatted_experience_level
Mid-Senior level    41489
Entry level         36708
NaN                 29409
Associate            9826
Director             3746
Internship           1449
Executive            1222
Name: count, dtype: int64

- work_type: 7 unique values
work_type
FULL_TIME     98814
CONTRACT      12117
PART_TIME      9696
TEMPORARY      1190
INTERNSHIP      983
VOLUNTEER       562
OTHER           487
Name: count, dtype: int64

- remote_allowed: 1 unique values
remote_allowed
NaN    108603
1.0     15246
Name: count, dtype: int64

- pay_period: 5 unique values
pay_period
NaN         87776
YEARLY      20628
HOURLY      14741
MONTHLY       518
WEEKLY        177
BIWEEKLY        9
Name: count, dtype: int64

- location: 8526 unique values
location
United States    8125
New York, NY     2756
Chicago, IL      1834
Houston, TX      1762
Dallas, TX       1383
Atlanta, GA      1363
Boston, MA       1176
Au

# STEP 2 — Cleaning & Preparing Dataset

In [10]:
# Remove exact duplicate job posts
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

In [11]:
# Clean text fields (title, description, skills)
import re

def clean_text(txt):
    if isinstance(txt, str):
        txt = txt.lower()
        txt = re.sub(r'\s+', ' ', txt)
        txt = re.sub(r'[^a-z0-9 ,.-]', '', txt)
    return txt

df['title'] = df['title'].apply(clean_text)
df['description'] = df['description'].apply(clean_text)
df['skills_desc'] = df['skills_desc'].apply(clean_text)

In [12]:
# Fill skills_desc smartly (98% missing → we must recreate it)

df['skills_desc'] = df['skills_desc'].fillna(df['title'] + " " + df['description'])

In [13]:
df.dropna(subset=['description'], inplace=True)

In [14]:
df['full_text'] = (
    df['title'].astype(str) + ". " + 
    df['skills_desc'].astype(str) + ". " + 
    df['description'].astype(str)
)

# STEP 3 — Build Semantic Embeddings (Core Intelligence Layer)

In [28]:
!pip install sentence-transformers

In [30]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [32]:
texts = df['full_text'].tolist()
embeddings = model.encode(texts, batch_size=64, show_progress_bar=True)
df['embedding'] = embeddings.tolist()

Batches:   0%|          | 0/1869 [00:00<?, ?it/s]

In [34]:
embeddings = model.encode(
    texts,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/3738 [00:00<?, ?it/s]

# STEP 4 — Create a Search Function (Cosine Similarity)

In [37]:
# Save embeddings into a NumPy array
import numpy as np
emb_matrix = np.vstack(df['embedding'].values)

In [39]:
# Create a function to get recommendations based on a user skill input
from sklearn.metrics.pairwise import cosine_similarity

def recommend_jobs(user_text, top_k=10):
    # 1. Convert user query to embedding
    user_emb = model.encode([user_text], normalize_embeddings=True)

    # 2. Compute cosine similarity with all job embeddings
    sims = cosine_similarity(user_emb, emb_matrix)[0]

    # 3. Get top job indices
    top_idx = sims.argsort()[-top_k:][::-1]

    # 4. Return results
    return df.iloc[top_idx][['title', 'skills_desc', 'location', 'formatted_experience_level', 'normalized_salary']]

In [41]:
# Test the recommender
recommend_jobs("python machine learning data science", top_k=5)

,title,skills_desc,location,formatted_experience_level,normalized_salary
66632,"senior machine learning engineer python, pyspa...","senior machine learning engineer python, pyspa...","New York, NY",Mid-Senior level,NaN
66607,"senior machine learning engineer python, pyspa...","senior machine learning engineer python, pyspa...","McLean, VA",Mid-Senior level,NaN
6406,python developer,python developer python developer - remote us ...,United States,NaN,NaN
79002,python developer,"python developer python developer - remote, u...",United States,NaN,NaN
50654,machine learning engineer- 10year exp,machine learning engineer- 10year exp . experi...,"Sunnyvale, CA",NaN,NaN


# STEP 5 — Build a Professional Ranking Formula

In [62]:
# Normalize numeric fields (experience, salary)
# Because similarity scores range from 0 to 1, but salary/experience don’t.
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

In [50]:
exp_map = {
    "internship": 0,
    "entry level": 1,
    "associate": 2,
    "mid-senior level": 3,
    "director": 4,
    "executive": 5
}

In [52]:
# Normalize text format
df['formatted_experience_level'] = (
    df['formatted_experience_level']
    .str.lower()
    .str.strip()
)

In [54]:
# Convert text → numeric
df['exp_level_num'] = df['formatted_experience_level'].map(exp_map)

In [56]:
# Handle missing experience level (default = entry level)
df['exp_level_num'] = df['exp_level_num'].fillna(1)

In [58]:
# Scale it
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

df['exp_norm'] = scaler.fit_transform(df[['exp_level_num']])

In [60]:
df['salary_norm'] = scaler.fit_transform(df[['normalized_salary']].fillna(0))

In [68]:
# Build the Final Ranking Score (Top-Tier Version)
from sklearn.metrics.pairwise import cosine_similarity

# User skills input (example)
user_text = "python machine learning sql data visualization"

# Convert user text to embedding
user_embedding = model.encode(
    user_text,
    normalize_embeddings=True
)

# Compute similarity with job embeddings
from sklearn.metrics.pairwise import cosine_similarity

df['similarity'] = cosine_similarity(
    user_embedding.reshape(1, -1),
    embeddings
).flatten()

In [70]:
# Add the ranking column
df['ranking_score'] = (
    0.7 * df['similarity'] +
    0.2 * df['exp_norm'] +
    0.1 * df['salary_norm']
)

In [72]:
# Get Top 10 Job Recommendations
results = df.sort_values('ranking_score', ascending=False).head(10)
results[['title', 'location', 'formatted_experience_level', 'similarity', 'ranking_score']]

,title,location,formatted_experience_level,similarity,ranking_score
66632,"senior machine learning engineer python, pyspa...","New York, NY",mid-senior level,0.510525,0.477368
66607,"senior machine learning engineer python, pyspa...","McLean, VA",mid-senior level,0.510525,0.477368
10427,senior sql developer,"Tampa, FL",mid-senior level,0.487931,0.461574
32780,data analyst,United States,mid-senior level,0.483276,0.458293
66098,biebusiness intelligence engineerbusiness inte...,"Seattle, WA",mid-senior level,0.464253,0.444977
62225,senior power bi developer,Dallas-Fort Worth Metroplex,mid-senior level,0.463808,0.444666
61261,"enterprise data engineer azure, python, sql","Boston, MA",mid-senior level,0.462476,0.443733
17441,vice president of data,United States,executive,0.347907,0.443593
25879,data scientist,"Austin, Texas Metropolitan Area",mid-senior level,0.457487,0.440261
79249,data science visualization engineer,"Cupertino, CA",entry level,0.558888,0.431222


In [81]:
df.to_pickle("processed_jobs.pkl")